# 21b — CD4 malignant burden (early vs advanced MF)

Loads nb21's saved T object (`skin_T_tcr_malig_v2.h5ad`), keeps **CD4 cells only** (MF + HC donors),
and summarises the ALICE malignancy call (`tcr_malignant_alice`) per sample. CPU-light.

Stages: **MF-early** {IA, IB, IIA} · **MF-advanced** {IIB, III, IV\*, advanced/late} · **HC**.

Figures: malignant vs total CD4 per sample · malignant fraction by stage · malignant count by stage ·
malignant fraction by disease.

In [ ]:
# ============================================================
# Parameters — load nb21's saved annotated T object + stage buckets.
# ============================================================
from pathlib import Path


def _resolve_nb_dir() -> Path:
    start = Path.cwd()
    for base in [start, *start.parents]:
        for sub in [Path("."), Path("notebooks/MF"), Path("scvi-tools-neural-nmf/notebooks/MF")]:
            cand = base / sub
            if cand.name == "MF" and (cand / "data").exists():
                return cand.resolve()
    raise FileNotFoundError(f"could not locate MF/data from {start}")


NB_DIR = _resolve_nb_dir(); print("NB_DIR =", NB_DIR)

# ---- input: nb21 output (cohort-filtered T cells + tcr_malignant_alice) ----
OUT_DIR = NB_DIR / "data" / "atlas_joint"
ADATA_MALIG_H5AD = OUT_DIR / "skin_T_tcr_malig_v2.h5ad"

CD4_T_TYPES = ["CD4"]
EARLY = {"IA", "IB", "IIA"}          # advanced = IIB|III|IV* + {advanced, late}

FIG_DIR = NB_DIR / "figures"; FIG_DIR.mkdir(exist_ok=True)

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc

np.random.seed(0)
plt.rcParams["figure.dpi"] = 120

In [ ]:
# Load nb21's saved annotated T object; keep CD4 cells only (incl. HC donors).
if not ADATA_MALIG_H5AD.exists():
    raise FileNotFoundError(f"{ADATA_MALIG_H5AD} missing — run 21_tcr_alice_neighborhood.ipynb first")
adata = sc.read_h5ad(ADATA_MALIG_H5AD)
adata = adata[adata.obs["cell_type_T2"].isin(CD4_T_TYPES)].copy()
print(adata)
print("tcr_malignant_alice:\n", adata.obs["tcr_malignant_alice"].value_counts())

In [ ]:
# stage_group bucket: HC / MF-early / MF-advanced / other (case-insensitive on disease_stage).
import re
_ADV = re.compile(r"IIB|III|IV", re.I)


def bucket(disease, stage):
    if str(disease).strip() == "HC":
        return "HC"
    su = str(stage).strip().upper()
    if su in EARLY or su == "EARLY":
        return "MF-early"
    if _ADV.search(su) or su in {"ADVANCED", "LATE"}:
        return "MF-advanced"
    return "other"


adata.obs["stage_group"] = [bucket(d, s) for d, s in
                            zip(adata.obs["disease"].astype(str), adata.obs["disease_stage"].astype(str))]
print(adata.obs.drop_duplicates("donor")["stage_group"].value_counts())

In [ ]:
# Per-donor CD4 summary: malignant burden vs sample size, with stage + disease.
per_donor = (adata.obs.groupby("donor", observed=True)
             .agg(n_tcells=("tcr_malignant_alice", "size"),
                  n_malignant=("tcr_malignant_alice", "sum"),
                  disease=("disease", "first"),
                  stage_group=("stage_group", "first"))
             .reset_index())
per_donor["prop_malignant"] = per_donor["n_malignant"] / per_donor["n_tcells"]

PAL = {"MF-early": "#4daf4a", "MF-advanced": "#e41a1c", "HC": "#377eb8", "other": "#999999"}
ORDER = ["MF-early", "MF-advanced", "HC"]


def box_strip(ax, df, group, value, order, ylabel=None, palette=None):
    data = [df.loc[df[group].eq(g), value].dropna().to_numpy() for g in order]
    ax.boxplot(data, labels=[str(g).replace("MF-", "") for g in order], showfliers=False, widths=0.6)
    for i, (g, arr) in enumerate(zip(order, data), 1):
        ax.scatter(np.random.normal(i, 0.06, len(arr)), arr, s=18,
                   color=(palette or {}).get(g, "#377eb8"), alpha=0.85, zorder=3)
    ax.set_ylabel(ylabel or value)


print(per_donor)

## Figures

Per-sample CD4 malignant burden. Stage palette `MF-early=green`, `MF-advanced=red`, `HC=blue`. Saved to `figures/`.

In [ ]:
# Fig 1 — malignant cells vs total CD4 T cells per sample.
fig, ax = plt.subplots(figsize=(5, 3))
for g in [*ORDER, "other"]:
    s = per_donor[per_donor["stage_group"].eq(g)]
    if len(s):
        ax.scatter(s["n_tcells"], s["n_malignant"], s=30, color=PAL[g], label=g, alpha=0.85)
mx = per_donor["n_tcells"].max()
ax.plot([0, mx], [0, mx], "k--", lw=0.8)
ax.set_xlabel("CD4 T cells in sample"); ax.set_ylabel("malignant cells")
ax.set_title("Malignant cells vs sample size"); ax.legend(frameon=False, fontsize=8)
fig.tight_layout(); fig.savefig(FIG_DIR / "cd4_malignant_vs_size.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Fig 2 — proportion malignant: early vs advanced vs HC.
d = per_donor[per_donor["stage_group"].isin(ORDER)]
fig, ax = plt.subplots(figsize=(5, 3))
box_strip(ax, d, "stage_group", "prop_malignant", ORDER,
          ylabel="malignant fraction of CD4", palette=PAL)
ax.set_title("Malignant fraction by MF stage")
fig.tight_layout(); fig.savefig(FIG_DIR / "cd4_prop_malignant_by_stage.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Fig 3 — number of malignant cells: early vs advanced vs HC.
fig, ax = plt.subplots(figsize=(5, 3))
box_strip(ax, d, "stage_group", "n_malignant", ORDER, ylabel="malignant cells", palette=PAL)
ax.set_title("Malignant cell count by MF stage")
fig.tight_layout(); fig.savefig(FIG_DIR / "cd4_n_malignant_by_stage.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Fig 4 — malignant fraction by disease type.
dis_order = [x for x in ["MF", "CTCL_other", "HC"] if x in set(per_donor["disease"])]
fig, ax = plt.subplots(figsize=(5, 3))
box_strip(ax, per_donor, "disease", "prop_malignant", dis_order, ylabel="malignant fraction of CD4")
ax.set_title("Malignant fraction by disease")
fig.tight_layout(); fig.savefig(FIG_DIR / "cd4_prop_malignant_by_disease.png", dpi=150, bbox_inches="tight")
plt.show()